# NB03 · Gold (Spain Utilities Pulse)

Lee las tablas del esquema `silver` y **reconstruye por completo** el modelo en estrella del esquema `gold` en cada ejecución (es pequeño y así no hay deriva entre capas).

| Tipo | Tabla Gold | Clave | Filas (25 nov 2026) |
|---|---|---|---|
| Dimensión | `gold.dim_fecha` | `fecha` | 1.096 |
| Dimensión | `gold.dim_serie_demanda` | `serie` | 1 |
| Dimensión | `gold.dim_punto_gas` (con `tipo_punto` y `es_suma_fisica`) | `point_key` | 15 |
| Dimensión | `gold.dim_direccion_gas` | `direction_key` | 2 |
| Dimensión | `gold.dim_ambito_agua` | `ambito` | 16 |
| Hecho | `gold.fact_demanda_diaria` | `serie`, `fecha` | 1.058 |
| Hecho | `gold.fact_flujo_gas_diario` | `fecha`, `point_key`, `direction_key` | 19.213 |
| Hecho | `gold.fact_almacenamiento_gas_diario` | `fecha` | 1.058 |
| Hecho | `gold.fact_reserva_agua_semanal` | `ambito`, `fecha` | 2.432 |
| Auxiliar | `gold.fact_frescura` | `tabla_silver` | 4 |

**Electricidad:** los valores de REE se pasan de MWh a GWh.

**Gas:** ENTSOG llega en kWh/d y se pasa a GWh. `fact_flujo_gas_diario` excluye valores atípicos (más de 10 veces el percentil 99 de su punto y dirección) y los lista al ejecutar; en la última ejecución se excluyó uno: VIP Pirineos (`ITP-00304`, salida) el 30 mar 2026, con 12.527 GWh frente a un percentil 99 de 170 GWh. Los puntos con `es_suma_fisica = False` (consumo agregado y punto virtual) no deben sumarse con el resto, porque ya contienen flujos de otros puntos.

**Agua:** se agrega la reserva de MITECO por ámbito (demarcación) y semana. La reserva del año anterior y la **media de 10 años** se calculan aquí, en Gold, por número de semana ISO; el modelo de Power BI solo las divide entre la capacidad. Silver guarda desde 2014, pero Gold publica desde 2024.

**Frescura:** `fact_frescura` guarda el último dato de cada tabla Silver, los días desde ese dato y si está al día: margen de 7 días para las fuentes diarias (REE, ENTSOG y AGSI+) y de 14 para la semanal (MITECO). Alimenta el panel de frescura del informe.

**Validaciones al final:** ninguna tabla vacía, ninguna clave repetida y ningún valor de un hecho sin su fila en la dimensión. Si algo falla, el notebook lanza un error para que el pipeline marque la actividad como fallida.

Requiere un Lakehouse con **esquemas habilitados** y asociado como predeterminado, y haber ejecutado NB02 (Silver).

In [ ]:
from pyspark.sql import functions as F, Window
import datetime as dt

SILVER = 'silver'
GOLD = 'gold'
HOY = dt.date.today()

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {GOLD}')
resumen = []

def guardar(df, nombre, claves):
    """Sobrescribe la tabla Gold completa y anota filas para el resumen."""
    full = f'{GOLD}.{nombre}'
    (df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(full))
    resumen.append({'tabla': full, 'claves': ', '.join(claves), 'filas': spark.table(full).count()})

ree = spark.table(f'{SILVER}.ree_demanda_diaria')
gas = spark.table(f'{SILVER}.entsog_flujo_fisico')
alm = spark.table(f'{SILVER}.agsi_almacenamiento')
print(f'Hoy: {HOY} | Silver: REE {ree.count()} | ENTSOG {gas.count()} | AGSI+ {alm.count()} filas')

In [ ]:
# dim_fecha: un día por fila desde el primer dato de Silver hasta el 31 de diciembre del año en curso
ini = (ree.select('fecha').unionByName(gas.select('fecha')).unionByName(alm.select('fecha'))
          .agg(F.min('fecha')).first()[0])
fin = dt.date(HOY.year, 12, 31)

MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
DIAS = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']
a_meses = F.array(*[F.lit(m) for m in MESES])
a_dias = F.array(*[F.lit(d) for d in DIAS])

dim_fecha = (spark.sql(f"SELECT explode(sequence(to_date('{ini}'), to_date('{fin}'), interval 1 day)) AS fecha")
    .withColumn('anio', F.year('fecha'))
    .withColumn('trimestre', F.quarter('fecha'))
    .withColumn('mes', F.month('fecha'))
    .withColumn('nombre_mes', F.element_at(a_meses, F.month('fecha')))
    .withColumn('anio_mes', F.date_format('fecha', 'yyyy-MM'))
    .withColumn('inicio_mes', F.trunc('fecha', 'MM'))
    .withColumn('semana_iso', F.weekofyear('fecha'))
    .withColumn('dia_semana_num', F.expr('weekday(fecha) + 1'))
    .withColumn('nombre_dia', F.element_at(a_dias, F.expr('weekday(fecha) + 1')))
    .withColumn('es_fin_de_semana', F.expr('weekday(fecha) >= 5')))

guardar(dim_fecha, 'dim_fecha', ['fecha'])

In [ ]:
# dim_serie_demanda: series de demanda de REE (real, programada, prevista...)
dim_serie = ree.select('serie').distinct()
guardar(dim_serie, 'dim_serie_demanda', ['serie'])

# dim_punto_gas: un punto de ENTSOG por clave; si el nombre ha cambiado con el tiempo, se queda el más reciente
w = F.row_number().over(Window.partitionBy('point_key').orderBy(F.col('fecha').desc()))
dim_punto = (gas.filter(F.col('point_label').isNotNull())
                .withColumn('_rn', w).filter('_rn = 1')
                .select('point_key', 'point_label'))
# puntos sin nombre en ningún registro: se usa la clave como etiqueta
sin_nombre = (gas.select('point_key').distinct().join(dim_punto, 'point_key', 'left_anti')
                 .withColumn('point_label', F.col('point_key')))
puntos = dim_punto.unionByName(sin_nombre)

# tipo_punto según el prefijo de la clave de ENTSOG. Los agregados y virtuales (FNC, VTP) ya contienen
# flujos de otros puntos: es_suma_fisica = False evita contar el mismo gas dos veces al sumar.
pref = F.substring('point_key', 1, 3)
puntos = (puntos
    .withColumn('tipo_punto', F.when(pref == 'ITP', 'Interconexión').when(pref == 'LNG', 'Planta GNL')
                               .when(pref == 'UGS', 'Almacenamiento').when(pref == 'FNC', 'Consumo agregado')
                               .when(pref == 'VTP', 'Punto virtual').otherwise('Otro'))
    .withColumn('es_suma_fisica', ~F.col('tipo_punto').isin('Consumo agregado', 'Punto virtual')))
guardar(puntos, 'dim_punto_gas', ['point_key'])

# dim_direccion_gas: entrada / salida del sistema
dim_dir = (gas.select('direction_key').distinct()
              .withColumn('direccion', F.when(F.lower('direction_key') == 'entry', 'Entrada')
                                         .when(F.lower('direction_key') == 'exit', 'Salida')
                                         .otherwise(F.col('direction_key'))))
guardar(dim_dir, 'dim_direccion_gas', ['direction_key'])

In [ ]:
# Hechos: se quitan las columnas técnicas de Silver (anio, mes, cargado_en) y se añaden valores en GWh
fact_demanda = (ree.select('fecha', 'serie', 'valor_mwh')
                   .withColumn('valor_gwh', F.col('valor_mwh') / 1000))
guardar(fact_demanda, 'fact_demanda_diaria', ['serie', 'fecha'])

# Gas: ENTSOG llega en kWh/d (validado). Se excluyen valores atípicos: los que superan 10 veces el
# percentil 99 de su punto y dirección (p. ej. VIP Pirineos exit el 2026-03-30: 12.527 GWh frente a un máximo normal de ~200).
# Gold se reconstruye cada vez: si ENTSOG corrige el dato, vuelve a entrar solo.
gas_g = gas.select('fecha', 'point_key', 'direction_key', 'valor', 'unidad')
p99 = (gas_g.groupBy('point_key', 'direction_key')
            .agg(F.percentile_approx('valor', 0.99).alias('p99')))
gas_g = gas_g.join(p99, ['point_key', 'direction_key'], 'left')
atipicos = gas_g.filter((F.col('p99') > 0) & (F.col('valor') > 10 * F.col('p99')))
n_atip = atipicos.count()
print(f'Valores atípicos excluidos de fact_flujo_gas_diario: {n_atip}')
if n_atip:
    display(atipicos.select('fecha', 'point_key', 'direction_key', (F.col('valor') / 1e6).alias('gwh'), (F.col('p99') / 1e6).alias('p99_gwh')))

fact_gas = (gas_g.filter(~((F.col('p99') > 0) & (F.col('valor') > 10 * F.col('p99'))))
                 .select('fecha', 'point_key', 'direction_key', 'valor', 'unidad')
                 .withColumn('valor_gwh', F.when(F.col('unidad').like('kWh%'), F.col('valor') / 1e6)))
guardar(fact_gas, 'fact_flujo_gas_diario', ['fecha', 'point_key', 'direction_key'])

fact_alm = alm.drop('anio', 'mes', 'cargado_en')
guardar(fact_alm, 'fact_almacenamiento_gas_diario', ['fecha'])

In [ ]:
# Agua: reserva semanal por ámbito (demarcación). Silver guarda desde 2014; Gold publica desde 2024.
VENTANA_GOLD = dt.date(2024, 1, 1)
emb = spark.table(f'{SILVER}.miteco_embalses_semanal')

ag = (emb.groupBy('ambito', 'fecha')
         .agg(F.sum('capacidad_hm3').alias('capacidad_hm3'), F.sum('reserva_hm3').alias('reserva_hm3'),
              F.countDistinct('embalse').alias('n_embalses'))
         .withColumn('anio', F.year('fecha')).withColumn('sem', F.weekofyear('fecha')))

hist = ag.select('ambito', 'sem', F.col('anio').alias('anio_h'), F.col('reserva_hm3').alias('res_h'))
cruce = ag.select('ambito', 'fecha', 'anio', 'sem').join(hist, ['ambito', 'sem'])
ant = (cruce.filter(F.col('anio_h') == F.col('anio') - 1)
            .groupBy('ambito', 'fecha').agg(F.avg('res_h').alias('reserva_anio_ant_hm3')))
m10 = (cruce.filter((F.col('anio_h') >= F.col('anio') - 10) & (F.col('anio_h') <= F.col('anio') - 1))
            .groupBy('ambito', 'fecha').agg(F.avg('res_h').alias('reserva_media10_hm3')))

fact_agua = (ag.join(ant, ['ambito', 'fecha'], 'left').join(m10, ['ambito', 'fecha'], 'left')
               .filter(F.col('fecha') >= F.lit(VENTANA_GOLD))
               .select('ambito', 'fecha', 'capacidad_hm3', 'reserva_hm3', 'reserva_anio_ant_hm3',
                       'reserva_media10_hm3', 'n_embalses'))
guardar(fact_agua, 'fact_reserva_agua_semanal', ['ambito', 'fecha'])

dim_ambito = fact_agua.select('ambito').distinct()
guardar(dim_ambito, 'dim_ambito_agua', ['ambito'])

In [ ]:
# fact_frescura: último dato de cada tabla Silver y si está al día (7 días para las diarias, 14 para la semanal de MITECO)
FUENTES = [('ree_demanda_diaria', 'REE · Demanda eléctrica', 'diaria', 7),
           ('entsog_flujo_fisico', 'ENTSOG · Flujo de gas Enagás', 'diaria', 7),
           ('agsi_almacenamiento', 'AGSI+ · Almacenamiento de gas', 'diaria', 7),
           ('miteco_embalses_semanal', 'MITECO · Reserva de embalses', 'semanal', 14)]

filas = []
for t, nombre, frec, umbral in FUENTES:
    ult = spark.table(f'{SILVER}.{t}').agg(F.max('fecha')).first()[0]
    dias = (HOY - ult).days if ult else None
    filas.append((t, nombre, frec, umbral, ult, dias, bool(dias is not None and dias <= umbral)))

fact_frescura = spark.createDataFrame(filas, 'tabla_silver string, fuente string, frecuencia string, umbral_dias int, '
                                             'ultimo_dato date, dias_desde int, al_dia boolean')
guardar(fact_frescura, 'fact_frescura', ['tabla_silver'])

In [ ]:
# Validaciones: tablas no vacías, claves no repetidas y hechos sin huérfanos respecto a sus dimensiones
fallos = []

CLAVES = {'dim_fecha': ['fecha'], 'dim_serie_demanda': ['serie'], 'dim_punto_gas': ['point_key'],
          'dim_direccion_gas': ['direction_key'], 'fact_demanda_diaria': ['serie', 'fecha'],
          'fact_flujo_gas_diario': ['fecha', 'point_key', 'direction_key'],
          'fact_almacenamiento_gas_diario': ['fecha'], 'fact_frescura': ['tabla_silver'],
          'dim_ambito_agua': ['ambito'], 'fact_reserva_agua_semanal': ['ambito', 'fecha']}
for t, claves in CLAVES.items():
    d = spark.table(f'{GOLD}.{t}')
    n, distintas = d.count(), d.select(*claves).distinct().count()
    if n == 0: fallos.append(f'{t} vacía')
    if n != distintas: fallos.append(f'{t} con claves repetidas ({n - distintas})')

RELACIONES = [('fact_demanda_diaria', 'fecha', 'dim_fecha', 'fecha'),
              ('fact_demanda_diaria', 'serie', 'dim_serie_demanda', 'serie'),
              ('fact_flujo_gas_diario', 'fecha', 'dim_fecha', 'fecha'),
              ('fact_flujo_gas_diario', 'point_key', 'dim_punto_gas', 'point_key'),
              ('fact_flujo_gas_diario', 'direction_key', 'dim_direccion_gas', 'direction_key'),
              ('fact_almacenamiento_gas_diario', 'fecha', 'dim_fecha', 'fecha'),
              ('fact_reserva_agua_semanal', 'fecha', 'dim_fecha', 'fecha'),
              ('fact_reserva_agua_semanal', 'ambito', 'dim_ambito_agua', 'ambito')]
for hecho, col, dim, dcol in RELACIONES:
    huerfanos = (spark.table(f'{GOLD}.{hecho}').select(col).distinct()
                 .join(spark.table(f'{GOLD}.{dim}').select(F.col(dcol).alias(col)), col, 'left_anti').count())
    if huerfanos: fallos.append(f'{hecho}.{col} con {huerfanos} valores sin fila en {dim}')

display(spark.createDataFrame(resumen))
display(spark.table(f'{GOLD}.fact_frescura'))
if fallos:
    raise RuntimeError(f'Validaciones Gold fallidas: {fallos}')
print('Gold OK: todas las validaciones pasan')